# 14 實戰案例 — 參考解答

松柏護理之家退伍軍人症迷你疫調報告的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || True
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy import stats

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150


## 題目 1：疫情摘要表

In [ ]:
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

n_total = len(df)
n_infected = int(df["infected"].sum())
n_deaths = int((df["outcome"] == "dead").sum())
n_hosp = int(df["hospitalized"].sum())
n_icu = int(df["icu_admission"].sum())

summary = pd.DataFrame([
    ["總住民數", n_total, ""],
    ["感染人數", n_infected, f"{n_infected/n_total:.1%}"],
    ["死亡人數", n_deaths, f"{n_deaths/n_infected:.1%} (CFR)"],
    ["住院人數", n_hosp, f"{n_hosp/n_infected:.1%} (住院率)"],
    ["ICU 人數", n_icu, f"{n_icu/n_hosp:.1%} (ICU/住院)"],
    ["侵襲率", f"{n_infected/n_total:.1%}", ""],
    ["致死率", f"{n_deaths/n_infected:.1%}", ""],
], columns=["指標", "數值", "比例"])

print("=== 疫情摘要表 ===")
print(summary.to_string(index=False))

## 題目 2：危險因子快速篩查

In [ ]:
factors = ["shower_use", "hydrotherapy_use", "comorbidity_copd", "immunosuppressed"]
results = []

for factor in factors:
    exposed_inf = int(df[(df[factor] == 1) & (df["infected"] == 1)].shape[0])
    exposed_n = int(df[df[factor] == 1].shape[0])
    unexposed_inf = int(df[(df[factor] == 0) & (df["infected"] == 1)].shape[0])
    unexposed_n = int(df[df[factor] == 0].shape[0])

    ar_exp = exposed_inf / exposed_n if exposed_n > 0 else 0
    ar_unexp = unexposed_inf / unexposed_n if unexposed_n > 0 else 0
    rr = ar_exp / ar_unexp if ar_unexp > 0 else float("inf")

    chi2, p, _, _ = stats.chi2_contingency(
        pd.crosstab(df[factor], df["infected"])
    )

    results.append({
        "factor": factor,
        "exposed_AR": f"{ar_exp:.1%}",
        "unexposed_AR": f"{ar_unexp:.1%}",
        "RR": f"{rr:.2f}",
        "p-value": f"{p:.4f}",
        "sig": "*" if p < 0.05 else "",
    })

rr_df = pd.DataFrame(results)
print("=== 危險因子 RR 比較表 ===")
print(rr_df.to_string(index=False))

max_rr = rr_df.loc[rr_df["RR"].astype(float).idxmax()]
print(f"\n→ RR 最大的因子: {max_rr['factor']} (RR = {max_rr['RR']})")
print("→ 淋浴使用是最強的暴露危險因子，與感染狀態有統計顯著關聯")

## 題目 3（挑戰題）：迷你 SitRep

In [ ]:
cases = df[df["infected"] == 1].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- 圖 1: 流行曲線 ---
daily = cases.groupby("symptom_onset_date").size()
full_range = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
daily = daily.reindex(full_range, fill_value=0)

axes[0].bar(daily.index, daily.values, color="steelblue", edgecolor="white")
peak = daily.idxmax()
axes[0].axvline(peak, color="red", linestyle="--", alpha=0.7)
axes[0].set_title(f"流行曲線 (高峰: {peak.strftime('%m/%d')})")
axes[0].set_ylabel("每日新增")
axes[0].tick_params(axis="x", rotation=45)

# --- 圖 2: 年齡分布 ---
for label, grp in df.groupby("infected"):
    tag = "感染" if label == 1 else "未感染"
    axes[1].hist(grp["age"], bins=15, alpha=0.6, label=tag, edgecolor="white")
axes[1].set_title("年齡分布")
axes[1].set_xlabel("年齡")
axes[1].legend()

# --- 圖 3: 樓層翼區侵襲率 ---
zone = df.groupby(["floor", "wing"])["infected"].agg(["sum", "count"]).reset_index()
zone["ar"] = zone["sum"] / zone["count"] * 100
zone["label"] = zone["floor"].astype(str) + "F-" + zone["wing"]
colors = ["#e74c3c" if ar > 50 else "steelblue" for ar in zone["ar"]]
axes[2].bar(zone["label"], zone["ar"], color=colors)
axes[2].axhline(50, color="red", linestyle="--", alpha=0.5)
axes[2].set_title("樓層翼區侵襲率")
axes[2].set_ylabel("侵襲率 (%)")
for i, row in zone.iterrows():
    axes[2].text(i, row["ar"] + 1, f"{row['ar']:.0f}%", ha="center", fontsize=9)

plt.suptitle("松柏護理之家退伍軍人症群聚 — 迷你 SitRep", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# 行動建議
top_zone = zone.sort_values("ar", ascending=False).iloc[0]
print("=" * 40)
print("  行動建議")
print("=" * 40)
print(f"  1. 優先處理區域: {top_zone['label']} (侵襲率 {top_zone['ar']:.1f}%)")
print(f"  2. 次要關注: 2F-A (54.5%) — 兩區共占大多數個案")
print(f"  3. 立即停用高風險區淋浴設施")
print(f"  4. 對 2F、3F 水管系統進行環境採檢")
print("=" * 40)

### 解讀

- **題目 1**：摘要表是疫調報告的第一頁，讓決策者快速掌握規模
- **題目 2**：RR 篩查可以快速找出最值得深入調查的暴露因子
  - 注意：crude RR 未調整干擾因子，需搭配 Ch05 分層分析和 Ch06 邏輯斯迴歸
- **題目 3**：好的 SitRep 一定要有「行動建議」——分析的目的是支持決策

恭喜完成最後的練習！你已經具備用 Python 進行疫情調查的核心技能。

## 題目 4：諾羅病毒宴會群聚迷你疫調（諾羅病毒情境）

一場宴會後爆發腸胃炎群聚。從賓客的食物暴露與發病 line list 做完整迷你疫調。

1. 畫流行曲線（發病時間），判斷傳播型態
2. 對每種食物計算侵襲率與風險比 RR
3. 找出 RR 最高的可疑食物並做卡方檢定
4. 寫一段結論：可疑感染源與流行曲線的意義

In [ ]:
# 諾羅病毒宴會群聚：150 位賓客的食物暴露與發病 line list
from epi_learning.metrics import attack_rate, risk_ratio
rng = np.random.default_rng(1404)
n = 150
foods = ["生蠔", "沙拉", "甜點", "湯品"]
ate = {f: rng.binomial(1, 0.5, n) for f in foods}
p_ill = (0.05 + 0.7 * ate["生蠔"]).clip(0, 1)     # 生蠔受汙染
ill = rng.binomial(1, p_ill)
onset_hr = np.where(ill == 1, rng.normal(32, 8, n).clip(6, 72), np.nan)  # 諾羅潛伏 ~24-48h
guests = pd.DataFrame({"guest_id": range(1, n + 1), "ill": ill,
                       **{f: ate[f] for f in foods}, "onset_hr": np.round(onset_hr, 0)})
print(f"宴會 {n} 人，發病 {ill.sum()} 人（{ill.mean():.1%}）")

sick = guests[guests["ill"] == 1]
fig, ax = plt.subplots(figsize=(7, 3.5))
bins = range(0, 78, 6)
ax.hist(sick["onset_hr"], bins=bins, color="#D97757", edgecolor="white")
ax.set_xlabel("發病時間（宴會後小時）"); ax.set_ylabel("病例數")
ax.set_title("諾羅病毒宴會群聚流行曲線"); plt.tight_layout(); plt.show()

print("各食物風險比：")
results = []
for f in foods:
    a = guests[(guests[f] == 1) & (guests.ill == 1)].shape[0]
    b = guests[(guests[f] == 1) & (guests.ill == 0)].shape[0]
    c = guests[(guests[f] == 0) & (guests.ill == 1)].shape[0]
    d = guests[(guests[f] == 0) & (guests.ill == 0)].shape[0]
    ar_e = attack_rate(a, a + b); ar_u = attack_rate(c, c + d)
    rr = risk_ratio(a, a + b, c, c + d)
    chi2, p, _, _ = stats.chi2_contingency([[a, b], [c, d]])
    results.append((f, ar_e, ar_u, rr, p))
    print(f"  {f}: 吃={ar_e:.1%} 沒吃={ar_u:.1%} RR={rr:.2f} p={p:.3g}")

culprit = max(results, key=lambda r: r[3])
print(f"\n【結論】最可疑感染源：{culprit[0]}（RR={culprit[3]:.2f}, p={culprit[4]:.3g}）。")
print("流行曲線為單峰、潛伏期集中 → 點源(point source)暴露，符合受汙染食物一次性汙染的型態。")

## 題目 5：COVID-19 職場群聚調查（COVID-19 情境）

某公司出現 COVID-19 群聚，懷疑一場全員大會是暴露事件。

1. 畫流行曲線（依發病日）
2. 算各部門侵襲率，找出最高風險部門
3. 計算「參加大會」的風險比 RR
4. 寫結論：大會是否為可疑暴露？

In [ ]:
# COVID-19 職場群聚：某公司 200 名員工，一場全員大會為可疑暴露
from epi_learning.metrics import risk_ratio
rng = np.random.default_rng(1405)
n = 200
dept = rng.choice(["業務", "研發", "行政", "客服"], n, p=[0.3, 0.3, 0.2, 0.2])
meeting = rng.binomial(1, np.where(dept == "業務", 0.9, 0.4))   # 業務多半有參加
p_inf = (0.03 + 0.35 * meeting).clip(0, 1)
infected = rng.binomial(1, p_inf)
onset_day = np.where(infected == 1, rng.integers(2, 10, n), -1)   # 會後第幾天發病
staff = pd.DataFrame({"emp_id": range(1, n + 1), "dept": dept,
                      "meeting": meeting, "infected": infected, "onset_day": onset_day})
print(f"公司 {n} 人，確診 {infected.sum()} 人；參加大會者 {meeting.sum()} 人")

curve = staff[staff.infected == 1].groupby("onset_day").size()
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(curve.index, curve.values, color="#D97757", edgecolor="white")
ax.set_xlabel("大會後天數"); ax.set_ylabel("確診數")
ax.set_title("COVID-19 職場群聚流行曲線"); plt.tight_layout(); plt.show()

by_dept = staff.groupby("dept").agg(n=("emp_id", "size"), cases=("infected", "sum"))
by_dept["attack_rate"] = (by_dept["cases"] / by_dept["n"]).round(3)
print(by_dept.sort_values("attack_rate", ascending=False).to_string())

a = staff[(staff.meeting == 1) & (staff.infected == 1)].shape[0]
b = staff[(staff.meeting == 1) & (staff.infected == 0)].shape[0]
c = staff[(staff.meeting == 0) & (staff.infected == 1)].shape[0]
d = staff[(staff.meeting == 0) & (staff.infected == 0)].shape[0]
rr = risk_ratio(a, a + b, c, c + d)
print(f"\n參加大會 RR = {rr:.2f}")
print(f"【結論】大會為可疑暴露事件（RR={rr:.2f}）；業務部參與率最高故侵襲率最高，")
print("建議追蹤大會接觸者、加強通風與篩檢。")

## 題目 6（挑戰題）：登革熱社區疫情 SitRep（登革熱情境）

某社區登革熱流行，你要產出一份情勢報告（SitRep）。

1. 畫全社區每週流行曲線
2. 算各行政區累計每十萬人發生率，排出熱區
3. 判斷疫情趨勢（上升／持平／下降）
4. 寫一份 3–5 句 SitRep：規模、熱區、趨勢、防治建議

In [ ]:
# 登革熱社區疫情：5 個行政區、10 週的每週病例與人口（挑戰題：寫一份 SitRep）
rng = np.random.default_rng(1406)
districts = ["安南區", "三民區", "北屯區", "板橋區", "中西區"]
pop = {"安南區": 190000, "三民區": 340000, "北屯區": 280000, "板橋區": 550000, "中西區": 78000}
weekly_rate = {"安南區": 3.0, "三民區": 1.2, "北屯區": 0.8, "板橋區": 0.6, "中西區": 1.4}  # /10萬/週
_rows = []
for wk in range(1, 11):
    growth = 1.0 + 0.15 * wk    # 疫情逐週上升
    for d in districts:
        cases = rng.poisson(weekly_rate[d] * growth * pop[d] / 100000)
        _rows.append({"epi_week": wk, "district": d, "cases": cases})
dengue = pd.DataFrame(_rows)
region_pop = pd.DataFrame({"district": districts, "population": [pop[d] for d in districts]})
print(f"登革熱：{dengue['cases'].sum()} 例，10 週 × {len(districts)} 區")

weekly = dengue.groupby("epi_week")["cases"].sum()
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(weekly.index, weekly.values, color="#D97757", edgecolor="white")
ax.set_xlabel("流行週"); ax.set_ylabel("每週病例數")
ax.set_title("登革熱社區每週流行曲線"); plt.tight_layout(); plt.show()

by_dist = dengue.groupby("district")["cases"].sum().reset_index()
by_dist = by_dist.merge(region_pop, on="district")
by_dist["rate_per_100k"] = (by_dist["cases"] / by_dist["population"] * 100000).round(1)
by_dist = by_dist.sort_values("rate_per_100k", ascending=False)
print(by_dist.to_string(index=False))

first_half = weekly.iloc[:5].sum(); second_half = weekly.iloc[5:].sum()
trend = "上升" if second_half > first_half * 1.1 else ("下降" if second_half < first_half * 0.9 else "持平")
top = by_dist.iloc[0]
print(f"\n【SitRep】本社區登革熱累計 {dengue['cases'].sum()} 例，橫跨 10 週。")
print(f"熱區為 {top['district']}（{top['rate_per_100k']}/10萬，全區最高）。")
print(f"每週病例呈{trend}趨勢（後 5 週 {second_half} 例 vs 前 5 週 {first_half} 例）。")
print("建議：於熱區加強孳生源清除與噴藥、擴大民眾衛教與就醫通報。")